# 📱 Hệ Thống Theo Dõi Giá iPhone
### Phân tích & Dự đoán giá — Dự án thực tế
---

## 🔷 PHẦN 1 — Load & Khám phá Data

In [ ]:
# === CELL 1: Import thư viện ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# Style đẹp hơn
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Import xong!')

In [ ]:
# === CELL 2: Load data ===
df = pd.read_csv('../data/iphone_prices.csv', parse_dates=['date'])

print('📊 TỔNG QUAN DATA:')
print(f'  Số dòng   : {len(df):,}')
print(f'  Số cột    : {len(df.columns)}')
print(f'  Thời gian : {df["date"].min().date()} → {df["date"].max().date()}')
print()
df.head(10)

In [ ]:
# === CELL 3: Thống kê cơ bản ===
print('📈 THỐNG KÊ GIÁ THEO MODEL:')
summary = df.groupby('model')['price'].agg(['min','max','mean','std']).round(0)
summary.columns = ['Thấp nhất', 'Cao nhất', 'Trung bình', 'Độ lệch chuẩn']

# Format số đẹp
for col in summary.columns:
    summary[col] = summary[col].apply(lambda x: f'{x:,.0f} đ')

display(summary)

In [ ]:
# === CELL 4: Kiểm tra missing values ===
print('🔍 KIỂM TRA DATA QUALITY:')
print(df.isnull().sum())
print()
print('📦 Phân bổ theo Store:')
print(df['store'].value_counts())
print()
print('📦 Phân bổ Stock Status:')
print(df['stock_status'].value_counts())

---
## 🔷 PHẦN 2 — Visualize Xu Hướng Giá

In [ ]:
# === CELL 5: Plot giá theo thời gian — 3 model ===
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

models = df['model'].unique()
colors = ['#ef4444', '#3b82f6', '#10b981']

for i, (model, color) in enumerate(zip(models, colors)):
    data = df[df['model'] == model].groupby('date')['price'].mean()
    
    axes[i].plot(data.index, data.values / 1e6, color=color, linewidth=1.5, alpha=0.8)
    axes[i].fill_between(data.index, data.values / 1e6, alpha=0.1, color=color)
    axes[i].set_ylabel('Giá (triệu đ)', fontsize=11)
    axes[i].set_title(model.replace('_', ' '), fontsize=12, fontweight='bold')
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}M'))

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=30)
fig.suptitle('📱 Xu Hướng Giá iPhone 2023–2024', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/price_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: data/price_trend.png')

In [ ]:
# === CELL 6: So sánh giá trung bình theo tháng ===
df['month'] = df['date'].dt.to_period('M')
monthly = df.groupby(['month', 'model'])['price'].mean().unstack()
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(ax=ax, marker='o', markersize=4, linewidth=2)
ax.set_title('Giá Trung Bình Theo Tháng', fontsize=13, fontweight='bold')
ax.set_ylabel('Giá (VNĐ)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=45)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === CELL 7: Giá theo Store — Boxplot ===
fig, ax = plt.subplots(figsize=(12, 5))

# Lấy model đắt nhất để so sánh store
target_model = 'iPhone_15_Pro_Max_256GB'
data_model = df[df['model'] == target_model]

sns.boxplot(
    data=data_model, x='store', y='price',
    palette='husl', ax=ax
)
ax.set_title(f'So sánh giá theo Store — {target_model.replace("_"," ")}',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Giá (VNĐ)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

print('💡 Store nào rẻ nhất?')
avg_by_store = data_model.groupby('store')['price'].mean().sort_values()
for store, price in avg_by_store.items():
    print(f'  {store:<20}: {price:,.0f} đ')

---
## 🔷 PHẦN 3 — Dự đoán Giá với Prophet

In [ ]:
# === CELL 8: Chuẩn bị data cho Prophet ===
# Prophet yêu cầu DataFrame với đúng 2 cột: 'ds' và 'y'

TARGET_MODEL = 'iPhone_15_Pro_Max_256GB'

# Lấy data của model, lấy giá trung bình mỗi ngày
prophet_df = (
    df[df['model'] == TARGET_MODEL]
    .groupby('date')['price']
    .mean()
    .reset_index()
    .rename(columns={'date': 'ds', 'price': 'y'})  # đổi tên cột
)

print(f'📊 Data Prophet cho: {TARGET_MODEL}')
print(f'Số ngày: {len(prophet_df)}')
print()
prophet_df.head()

In [ ]:
# === CELL 9: Train model Prophet ===
print('🔧 Đang train Prophet...')

m = Prophet(
    yearly_seasonality=True,    # có seasonality theo năm
    weekly_seasonality=False,   # giá iPhone ít phụ thuộc ngày trong tuần
    daily_seasonality=False,
    changepoint_prior_scale=0.3,  # cho phép trend thay đổi linh hoạt
    interval_width=0.95           # độ tin cậy 95%
)

m.fit(prophet_df)
print('✅ Train xong!')

In [ ]:
# === CELL 10: Dự đoán 90 ngày tới ===
FORECAST_DAYS = 90

future = m.make_future_dataframe(periods=FORECAST_DAYS)
forecast = m.predict(future)

print(f'🔮 Dự đoán {FORECAST_DAYS} ngày tới:')
print()

# Lấy kết quả dự đoán (90 ngày cuối)
future_only = forecast.tail(FORECAST_DAYS)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
future_only.columns = ['Ngày', 'Dự đoán', 'Thấp nhất', 'Cao nhất']

# Format số
for col in ['Dự đoán', 'Thấp nhất', 'Cao nhất']:
    future_only[col] = future_only[col].apply(lambda x: f'{x:,.0f} đ')

display(future_only.head(15))

In [ ]:
# === CELL 11: Plot forecast ===
fig, ax = plt.subplots(figsize=(14, 6))

# Data thực tế
ax.plot(prophet_df['ds'], prophet_df['y'] / 1e6,
        color='#374151', linewidth=1.2, label='Giá thực tế', alpha=0.8)

# Đường dự đoán
ax.plot(forecast['ds'], forecast['yhat'] / 1e6,
        color='#3b82f6', linewidth=2, label='Dự đoán', linestyle='--')

# Vùng uncertainty
ax.fill_between(
    forecast['ds'],
    forecast['yhat_lower'] / 1e6,
    forecast['yhat_upper'] / 1e6,
    alpha=0.15, color='#3b82f6', label='Khoảng tin cậy 95%'
)

# Đường phân cách thực tế / dự đoán
split_date = prophet_df['ds'].max()
ax.axvline(x=split_date, color='#ef4444', linestyle=':', linewidth=1.5, label='Hôm nay')

ax.set_title(f'Dự Đoán Giá {TARGET_MODEL.replace("_", " ")} — 90 Ngày Tới',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Giá (triệu đ)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}M'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%Y'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../data/price_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: data/price_forecast.png')

In [ ]:
# === CELL 12: Plot components (trend + seasonality) ===
fig = m.plot_components(forecast)
fig.suptitle('Phân tích thành phần: Trend & Seasonality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Giải thích:')
print('  trend     → xu hướng giá dài hạn (tăng/giảm tổng thể)')
print('  yearly    → biến động theo mùa trong năm (Tết, mùa ra model mới...)')

---
## 🔷 PHẦN 4 — Insight & Kết luận

In [ ]:
# === CELL 13: Tóm tắt kết quả dự đoán ===
last_actual_price = prophet_df['y'].iloc[-1]
forecast_90d = forecast[forecast['ds'] > split_date]

min_forecast = forecast_90d['yhat'].min()
max_forecast = forecast_90d['yhat'].max()
end_forecast  = forecast_90d['yhat'].iloc[-1]

change_pct = (end_forecast - last_actual_price) / last_actual_price * 100

print('=' * 55)
print(f'  📱 MODEL : {TARGET_MODEL.replace("_", " ")}')
print('=' * 55)
print(f'  Giá hiện tại       : {last_actual_price:>15,.0f} đ')
print(f'  Dự đoán thấp nhất  : {min_forecast:>15,.0f} đ')
print(f'  Dự đoán cao nhất   : {max_forecast:>15,.0f} đ')
print(f'  Dự đoán sau 90 ngày: {end_forecast:>15,.0f} đ')
print(f'  Thay đổi dự kiến   : {change_pct:>+14.2f} %')
print('=' * 55)

if change_pct < 0:
    print(f'  📉 Giá sẽ GIẢM ~{abs(change_pct):.1f}% trong 90 ngày tới')
else:
    print(f'  📈 Giá sẽ TĂNG ~{change_pct:.1f}% trong 90 ngày tới')

In [ ]:
# === CELL 14: Thử đổi model khác — tự thực hành ===
# TODO: thay tên model bên dưới và chạy lại từ Cell 8

# Danh sách model có trong data:
print('Các model có sẵn trong data:')
for m_name in df['model'].unique():
    avg = df[df['model'] == m_name]['price'].mean()
    print(f'  → {m_name:<35} giá TB: {avg:,.0f} đ')

print()
print('💡 Thử đổi TARGET_MODEL ở Cell 8 thành một model khác và chạy lại!')